In [1]:
import math
import random
import copy
import networkx as nx
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict


random.seed(42)
np.random.seed(42)



In [2]:
print("\n[1/6] Importazione grafo")
G = nx.read_edgelist("Dataset/facebook_combined.txt")
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
tutti_i_gradi = [grado for nodo, grado in G.degree()]
grado_massimo = max(tutti_i_gradi)
grado_minimo = min(tutti_i_gradi)
print(f"      Nodi: {n_nodes}, Archi: {n_edges}")
print(f"      Grado medio: {2*n_edges/n_nodes:.2f}")
print(f"      Grado massimo: {grado_massimo}")
print(f"      Grado minimo: {grado_minimo}")
print(f"      Clustering medio: {nx.average_clustering(G):.4f}")



[1/6] Importazione grafo
      Nodi: 4039, Archi: 88234
      Grado medio: 43.69
      Grado massimo: 1045
      Grado minimo: 1
      Clustering medio: 0.6055


In [17]:
"""Visualizza proprietà del grafo."""
out_path = "/Users/giuseppepiosorrentino/RetiSocialiProj/Results/WTSS/"
degrees = [d for _, d in G.degree()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(degrees, bins=30, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].set_title("Distribuzione dei gradi", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Grado", fontsize=11)
axes[0].set_ylabel("Frequenza", fontsize=11)
axes[0].grid(True, alpha=0.3)

# Draw small version of the graph
pos = nx.spring_layout(G, seed=42, k=0.5)
nx.draw_networkx(G, pos=pos, ax=axes[1], node_size=20, node_color="#2196F3",
                   edge_color="#BBBBBB", with_labels=False, alpha=0.8, width=0.5)
axes[1].set_title("Visualizzazione della rete", fontsize=13, fontweight="bold")
axes[1].axis("off")

fig.suptitle(f"Rete: n={G.number_of_nodes()}, m={G.number_of_edges()}, "
             f"<k>={2*G.number_of_edges()/G.number_of_nodes():.1f}",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Salvato: {out_path}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/giuseppepiosorrentino/RetiSocialiProj/Results/WTSS/.png'

In [3]:
import random
import math

def cost_random(G, low=1, high=5, seed=42):
    rng = random.Random(seed)
    return {v: rng.randint(low, high) for v in G.nodes()}

def cost_half_degree(G):
    return {v: max(1, math.ceil(G.degree(v) / 2)) for v in G.nodes()}

#VALUTIAMO, è OPZIONALE
def cost_custom(G):
    clust = nx.clustering(G)
    return {v: max(1, round(1 + clust[v] * 10)) for v in G.nodes()}

# Istanzia le tre funzioni costo sul grafo G
c_random = cost_random(G)
c_half   = cost_half_degree(G)
#OPZIONALE
c_custom = cost_custom(G)

print("Esempio costi (primi 5 nodi):")
for v in list(G.nodes())[:5]:
    print(f"  nodo {v}: random={c_random[v]}, d/2={c_half[v]}, custom={c_custom[v]}")



Esempio costi (primi 5 nodi):
  nodo 0: random=1, d/2=174, custom=1
  nodo 1: random=1, d/2=9, custom=5
  nodo 2: random=3, d/2=5, custom=10
  nodo 3: random=2, d/2=9, custom=7
  nodo 4: random=2, d/2=5, custom=10


In [4]:
costo_totale_random = sum(c_random.values())
costo_totale_half   = sum(c_half.values())
costo_totale_custom = sum(c_custom.values())
k_random = int(0.10 * costo_totale_random)
k_half   = int(0.10 * costo_totale_half)
k_custom = int(0.10 * costo_totale_custom)

print(f"Costo totale random: {costo_totale_random}")
print(f"Costo totale half:   {costo_totale_half}")
print(f"Costo totale custom: {costo_totale_custom}")

Costo totale random: 12173
Costo totale half:   89243
Costo totale custom: 28467


In [5]:
from functools import lru_cache
import numpy as np

# Pre-calcola strutture del grafo una volta sola
def precompute_graph(G):
    neighbors = {v: set(G.neighbors(v)) for v in G.nodes()}
    degrees   = {v: G.degree(v) for v in G.nodes()}
    thresholds = {v: math.ceil(degrees[v] / 2) for v in G.nodes()}
    return neighbors, degrees, thresholds

neighbors, degrees, thresholds = precompute_graph(G)

def f1(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        overlap = len(neighbors[v] & S_set)
        val += min(overlap, thresholds[v])
    return val

def f2(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        overlap = len(neighbors[v] & S_set)
        t = thresholds[v]
        # somma aritmetica invece del ciclo: t + (t-1) + ... 
        if overlap > 0:
            effective = min(overlap, t)
            val += effective * t - (effective * (effective - 1)) // 2
    return val

def f3(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        dv = degrees[v]
        if dv == 0:
            continue
        overlap = len(neighbors[v] & S_set)
        t = thresholds[v]
        for i in range(1, overlap + 1):
            denom = dv - i + 1
            if denom > 0:
                val += max((t - i + 1) / denom, 0)
    return val

In [6]:
def majority_cascade(G, S):
    """
    Simula il processo di attivazione con regola Majority.
    Un nodo v si attiva se almeno metà dei suoi vicini sono già attivi.
    Restituisce Inf[S] = insieme di tutti i nodi attivati.
    """
    infected = set(S)
    
    while True:
        new_infected = set()
        for v in G.nodes():
            if v not in infected:
                dv = G.degree(v)
                if dv == 0:
                    continue
                neighbors_infected = sum(1 for u in G.neighbors(v) if u in infected)
                if neighbors_infected >= math.ceil(dv / 2):
                    new_infected.add(v)
        
        if not new_infected:  # nessun nuovo nodo attivato: la cascata si ferma
            break
        infected |= new_infected
    
    return infected

In [7]:
from functools import lru_cache

def marginal_gain(G, v, S_current, fi_func):
    """
    Ottimizzazione: invece di ricalcolare fi su tutto il grafo,
    guarda solo i vicini di v che cambiano.
    """
    S_set = frozenset(S_current)
    val = 0
    # Solo i vicini di v sono influenzati dall'aggiunta di v
    for u in neighbors[v]:
        overlap_before = len(neighbors[u] & S_set)
        overlap_after  = overlap_before + (1 if v not in S_set else 0)
        
        if fi_func == f1:
            val += min(overlap_after, thresholds[u]) - min(overlap_before, thresholds[u])
        elif fi_func == f2:
            t = thresholds[u]
            def contrib(ov):
                effective = min(ov, t)
                return effective * t - (effective * (effective - 1)) // 2
            val += contrib(overlap_after) - contrib(overlap_before)
        elif fi_func == f3:
            dv = degrees[u]
            if dv == 0:
                continue
            t = thresholds[u]
            before = sum(max((t-i+1)/( dv-i+1), 0) for i in range(1, overlap_before+1))
            after  = sum(max((t-i+1)/(dv-i+1), 0)  for i in range(1, overlap_after+1))
            val += after - before
    return val


In [8]:
def wtss_maximal(G, k, c):
    """
    WTSS adattato per restituire il seed set massimale con costo <= k.
    Threshold t(v) = ceil(d(v)/2) per ogni nodo v.
    
    Logica:
    - Case 1: v ha threshold 0 → viene attivato dai vicini, non serve nel seed
    - Case 2: v non può essere attivato dai vicini rimasti → va nel seed
    - Case 3: tutti attivabili → rimuovi il nodo "meno urgente" dal grafo
    """
    # Inizializzazione
    delta     = {v: degrees[v] for v in G.nodes()}        # grado corrente
    k_thresh  = {v: thresholds[v] for v in G.nodes()}     # threshold corrente
    neigh     = {v: set(neighbors[v]) for v in G.nodes()} # vicini correnti
    U         = set(G.nodes())
    S         = set()
    current_cost = 0

    while U:
        # Case 1: esiste v con threshold == 0 → attivato dai vicini
        case1 = next((v for v in U if k_thresh[v] == 0), None)
        if case1 is not None:
            v = case1
            for u in neigh[v]:
                if u in U:
                    k_thresh[u] = max(0, k_thresh[u] - 1)
                    delta[u] -= 1
                    neigh[u].discard(v)
            neigh[v] = set()
            U.remove(v)
            continue

        # Case 2: delta(v) < k(v) → non può essere attivato, va nel seed
        case2 = next((v for v in U if delta[v] < k_thresh[v]), None)
        if case2 is not None:
            v = case2
            if current_cost + c[v] > k:
                return S  # budget esaurito
            S.add(v)
            current_cost += c[v]
            for u in neigh[v]:
                if u in U:
                    k_thresh[u] = max(0, k_thresh[u] - 1)
                    delta[u] -= 1
                    neigh[u].discard(v)
            neigh[v] = set()
            U.remove(v)
            continue

        # Case 3: tutti attivabili → rimuovi il nodo meno urgente
        v = max(U, key=lambda u: (c[u] * k_thresh[u]) / (delta[u] * (delta[u] + 1)) 
                if delta[u] > 0 else float('inf'))
        for u in neigh[v]:
            if u in U:
                delta[u] -= 1
                neigh[u].discard(v)
        neigh[v] = set()
        U.remove(v)

    return S

In [9]:
S_wtss = wtss_maximal(G, k_half, c_half)
inf_wtss = majority_cascade(G, S_wtss)

print(f"Seed set WTSS: {len(S_wtss)} nodi, costo={sum(c_half[u] for u in S_wtss)}")
print(f"|Inf[G, S_wtss]| = {len(inf_wtss)} / {G.number_of_nodes()}")

Seed set WTSS: 390 nodi, costo=8902
|Inf[G, S_wtss]| = 1725 / 4039


In [10]:
S_wtss_f2 = wtss_maximal(G, k_random, c_random)
print(f"Seed set WTSS-f2: {len(S_wtss_f2)} nodi, costo={sum(c_random[u] for u in S_wtss_f2)}")
print(f"|Inf[G, S_wtss_f2]| = {len(majority_cascade(G, S_wtss_f2))}")

Seed set WTSS-f2: 766 nodi, costo=1216
|Inf[G, S_wtss_f2]| = 3484


In [11]:
S_wtss_f3 = wtss_maximal(G, k_custom, c_custom)
print(f"Seed set WTSS-f3: {len(S_wtss_f3)} nodi, costo={sum(c_custom[u] for u in S_wtss_f3)}")
print(f"|Inf[G, S_wtss_f3]| = {len(majority_cascade(G, S_wtss_f3))}")

Seed set WTSS-f3: 500 nodi, costo=2842
|Inf[G, S_wtss_f3]| = 3127


In [12]:
# Funzione per rimuovere x archi casuali da G
def remove_edges(G, x, seed=42):
    G2 = G.copy()
    edges = list(G2.edges())
    rng = random.Random(seed)
    to_remove = rng.sample(edges, min(x, len(edges)))
    G2.remove_edges_from(to_remove)
    return G2

In [13]:
def remove_vertices(G, y, seed=42):
    G2 = G.copy()
    nodes = list(G2.nodes())
    rng = random.Random(seed)
    to_remove = rng.sample(nodes, min(y, len(nodes)))
    G2.remove_nodes_from(to_remove)
    return G2

GREEDY CON RANDOM

In [16]:
# Aggiungi punti intermedi nel range interessante
percentages = [0.05, 0.06, 0.07, 0.08, 0.09, 
               0.10, 0.12, 0.15, 0.20, 0.25, 
               0.30, 0.40, 0.50]
budgets_random = [int(p * sum(c_random.values())) for p in percentages]

inf_wtss_c1 = []

for k in budgets_random:
    S_f1 = wtss_maximal(G, k, c_random)
    inf_wtss_c1.append(len(majority_cascade(G, S_f1)))
    print(f"k={k:5d} | f1={inf_wtss_c1[-1]:4d} ")
print("\nHALF: ")


k=  608 | f1=1464 
k=  730 | f1=1830 
k=  852 | f1=2303 
k=  973 | f1=2866 
k= 1095 | f1=3234 
k= 1217 | f1=3484 
k= 1460 | f1=4039 
k= 1825 | f1=4039 
k= 2434 | f1=4039 
k= 3043 | f1=4039 
k= 3651 | f1=4039 
k= 4869 | f1=4039 
k= 6086 | f1=4039 

HALF: 
k= 4462 | f2= 584 
k= 5354 | f2= 598 
k= 6247 | f2= 609 
k= 7139 | f2= 622 
k= 8031 | f2= 636 
k= 8924 | f2=1725 
k=10709 | f2=1979 
k=13386 | f2=3266 
k=17848 | f2=3437 
k=22310 | f2=3488 
k=26772 | f2=4039 
k=35697 | f2=4039 
k=44621 | f2=4039 

CUSTOM: 
k= 1423 | f3=1140 
k= 1708 | f3=1387 
k= 1992 | f3=1611 
k= 2277 | f3=1910 
k= 2562 | f3=2338 
k= 2846 | f3=3127 
k= 3416 | f3=3276 
k= 4270 | f3=4039 
k= 5693 | f3=4039 
k= 7116 | f3=4039 
k= 8540 | f3=4039 
k=11386 | f3=4039 
k=14233 | f3=4039 


In [19]:
plt.figure(figsize=(9, 5))

plt.plot(percentages, inf_wtss_c1, color="#2196F3", marker="o", linewidth=2, label="WTSS-c1")

plt.axhline(y=G.number_of_nodes(), color="gray", linestyle="--", alpha=0.5, label=f"|V|={G.number_of_nodes()}")
plt.xlabel("Percentuale del costo totale")
plt.ylabel("|Inf[G, S]|")
plt.title("WTSS — |Inf[G,S]| al variare di k (c_random)") 
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "wtss_random", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/8z/67y4_fv93g3dgx30jl7w8ss40000gn/T/ipykernel_1844/549816842.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
# Seed set di riferimento calcolati con k=1217
k_random = 1095
k_half = 13386
k_custom = 2846
# Ripristina le strutture usate da wtss_maximal se "degrees" è stato sovrascritto (es. lista)
if not isinstance(degrees, dict):
    neighbors, degrees, thresholds = precompute_graph(G)

S_ref_c1 = wtss_maximal(G, k_random, c_random)
S_ref_c2 = wtss_maximal(G, k_half, c_half)
S_ref_c3 = wtss_maximal(G, k_custom, c_custom)
# Range di archi da rimuovere
x_range = list(range(0, 20000, 1000))

inf_c1_edges   = []
inf_c2_edges   = []
inf_c3_edges   = []


for x in x_range:
    G2 = remove_edges(G, x)
    inf_c1_edges.append(len(majority_cascade(G2, S_ref_c1)))
    inf_c2_edges.append(len(majority_cascade(G2, S_ref_c2)))
    inf_c3_edges.append(len(majority_cascade(G2, S_ref_c3)))
    print(f"x={x:5d} | c1={inf_c1_edges[-1]:4d} | c2={inf_c2_edges[-1]:4d} | c3={inf_c3_edges[-1]:4d} |")

x=    0 | c1=3234 | c2=3266 | c3=3127 |
x= 1000 | c1=3240 | c2=2346 | c3=3127 |
x= 2000 | c1=3239 | c2=1131 | c3=3127 |
x= 3000 | c1=3239 | c2=1308 | c3=3117 |
x= 4000 | c1=3239 | c2=1297 | c3=3119 |
x= 5000 | c1=3237 | c2=1305 | c3=3121 |
x= 6000 | c1=3236 | c2=2136 | c3=3127 |
x= 7000 | c1=3237 | c2=1730 | c3=3135 |
x= 8000 | c1=3237 | c2= 894 | c3=3146 |
x= 9000 | c1=3237 | c2= 912 | c3=3147 |
x=10000 | c1=3234 | c2=1303 | c3=3146 |
x=11000 | c1=3235 | c2= 989 | c3=3145 |
x=12000 | c1=3245 | c2= 978 | c3=3131 |
x=13000 | c1=3239 | c2= 978 | c3=3117 |
x=14000 | c1=3236 | c2=1477 | c3=3118 |
x=15000 | c1=3238 | c2=1497 | c3=3119 |
x=16000 | c1=3222 | c2= 972 | c3=3117 |
x=17000 | c1=3219 | c2= 987 | c3=3114 |
x=18000 | c1=3216 | c2=1427 | c3=3112 |
x=19000 | c1=3215 | c2= 991 | c3=3099 |


In [28]:
plt.figure(figsize=(9, 5))

plt.plot(x_range, inf_c1_edges,   color="#E91E63", marker="s", linewidth=2, label="WTSS-Crandom")
plt.plot(x_range, inf_c2_edges,   color="#FF9800", marker="D", linewidth=2, label="WTSS-Chalf")
plt.plot(x_range, inf_c3_edges,   color="#9C27B0", marker="^", linewidth=2, label="WTSS-Ccustom")

# Linea di riferimento: |Inf| sul grafo originale senza rimozioni
plt.axhline(y=3234, color="#E91E63", linestyle="--", alpha=0.3)
plt.axhline(y=3266, color="#FF9800", linestyle="--", alpha=0.3)
plt.axhline(y=3127, color="#9C27B0", linestyle="--", alpha=0.3)

plt.xlabel("Archi rimossi (x)")
plt.ylabel("|Inf[G', S]|")
plt.title("Step 2 — Effetto rimozione archi ")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "edge_removal.png", dpi=150, bbox_inches="tight")


In [41]:
# Seed set di riferimento calcolati con k=1217
k_random = 1095
k_half = 13386
k_custom = 2846
# Ripristina le strutture usate da wtss_maximal se "degrees" è stato sovrascritto (es. lista)
if not isinstance(degrees, dict):
    neighbors, degrees, thresholds = precompute_graph(G)

S_ref_c1 = wtss_maximal(G, k_random, c_random)
S_ref_c2 = wtss_maximal(G, k_half, c_half)
S_ref_c3 = wtss_maximal(G, k_custom, c_custom)
# Range di archi da rimuovere
y_range = list(range(0, 1000, 50))

inf_c1_edges   = []
inf_c2_edges   = []
inf_c3_edges   = []


for y in y_range:
    G2 = remove_vertices(G, y)
    inf_c1_edges.append(len(majority_cascade(G2, S_ref_c1)))
    inf_c2_edges.append(len(majority_cascade(G2, S_ref_c2)))
    inf_c3_edges.append(len(majority_cascade(G2, S_ref_c3)))
    print(f"y={y:5d} | c1={inf_c1_edges[-1]:4d} | c2={inf_c2_edges[-1]:4d} | c3={inf_c3_edges[-1]:4d} |")

y=    0 | c1=3234 | c2=3266 | c3=3127 |
y=   50 | c1=3206 | c2=1391 | c3=2909 |
y=  100 | c1=3169 | c2=2093 | c3=2875 |
y=  150 | c1=3055 | c2=1363 | c3=2846 |
y=  200 | c1=3117 | c2=1343 | c3=2815 |
y=  250 | c1=3026 | c2= 999 | c3=2652 |
y=  300 | c1=2993 | c2= 976 | c3=2624 |
y=  350 | c1=2939 | c2=1277 | c3=2596 |
y=  400 | c1=2906 | c2=1262 | c3=2589 |
y=  450 | c1=2877 | c2=1381 | c3=2563 |
y=  500 | c1=2861 | c2=1371 | c3=2651 |
y=  550 | c1=2825 | c2=1081 | c3=2612 |
y=  600 | c1=2796 | c2= 962 | c3=2585 |
y=  650 | c1=2766 | c2= 928 | c3=2554 |
y=  700 | c1=2736 | c2= 929 | c3=2523 |
y=  750 | c1=2705 | c2= 906 | c3=2489 |
y=  800 | c1=2642 | c2= 950 | c3=2428 |
y=  850 | c1=2618 | c2= 891 | c3=2387 |
y=  900 | c1=2595 | c2= 881 | c3=2358 |
y=  950 | c1=2563 | c2= 882 | c3=2339 |


In [42]:
plt.figure(figsize=(9, 5))

plt.plot(y_range, inf_c1_edges,   color="#E91E63", marker="s", linewidth=2, label="WTSS-Crandom")
plt.plot(y_range, inf_c2_edges,   color="#FF9800", marker="D", linewidth=2, label="WTSS-Chalf")
plt.plot(y_range, inf_c3_edges,   color="#9C27B0", marker="^", linewidth=2, label="WTSS-Ccustom")

# Linea di riferimento: |Inf| sul grafo originale senza rimozioni
plt.axhline(y=3234, color="#E91E63", linestyle="--", alpha=0.3)
plt.axhline(y=3266, color="#FF9800", linestyle="--", alpha=0.3)
plt.axhline(y=3127, color="#9C27B0", linestyle="--", alpha=0.3)

plt.xlabel("Nodi rimossi (x)")
plt.ylabel("|Inf[G', S]|")
plt.title("Step 2 — Effetto rimozione nodi ")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "node_removal.png", dpi=150, bbox_inches="tight")


GREEDY CON HALF

In [36]:
# Aggiungi punti intermedi nel range interessante
percentages = [0.05, 0.06, 0.07, 0.08, 0.09, 
               0.10, 0.12, 0.15, 0.20, 0.25, 
               0.30, 0.40, 0.50]
budgets_half = [int(p * sum(c_half.values())) for p in percentages]

inf_wtss = []


for k in budgets_half:
    S_f1 = wtss_maximal(G, k, c_half)
    
    
    inf_wtss.append(len(majority_cascade(G, S_f1)))
    
    print(f"k={k:5d} | wtss={inf_wtss[-1]:4d}")

k= 4462 | wtss= 584
k= 5354 | wtss= 598
k= 6247 | wtss= 609
k= 7139 | wtss= 622
k= 8031 | wtss= 636
k= 8924 | wtss=1725
k=10709 | wtss=1979
k=13386 | wtss=3266
k=17848 | wtss=3437
k=22310 | wtss=3488
k=26772 | wtss=4039
k=35697 | wtss=4039
k=44621 | wtss=4039


In [37]:
plt.figure(figsize=(9, 5))

plt.plot(percentages, inf_wtss, color="#2196F3", marker="o", linewidth=2, label="WTSS")

plt.axhline(y=G.number_of_nodes(), color="gray", linestyle="--", alpha=0.5, label=f"|V|={G.number_of_nodes()}")
plt.xlabel("Percentuale del costo totale")
plt.ylabel("|Inf[G, S]|")
plt.title("WTSS — |Inf[G,S]| al variare di k (c_half)") 
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "wtss_half", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/8z/67y4_fv93g3dgx30jl7w8ss40000gn/T/ipykernel_1844/3602283628.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


custom

In [34]:
# Aggiungi punti intermedi nel range interessante
percentages = [0.05, 0.06, 0.07, 0.08, 0.09, 
               0.10, 0.12, 0.15, 0.20, 0.25, 
               0.30, 0.40, 0.50]
budgets_custom = [int(p * sum(c_custom.values())) for p in percentages]

inf_wtss = []


for k in budgets_custom:
    S_f1 = wtss_maximal(G, k, c_custom)
   
    
    inf_wtss.append(len(majority_cascade(G, S_f1)))
    
    print(f"k={k:5d} | wtss={inf_wtss[-1]:4d}")

k= 1423 | wtss=1140
k= 1708 | wtss=1387
k= 1992 | wtss=1611
k= 2277 | wtss=1910
k= 2562 | wtss=2338
k= 2846 | wtss=3127
k= 3416 | wtss=3276
k= 4270 | wtss=4039
k= 5693 | wtss=4039
k= 7116 | wtss=4039
k= 8540 | wtss=4039
k=11386 | wtss=4039
k=14233 | wtss=4039


In [35]:
plt.figure(figsize=(9, 5))

plt.plot(percentages, inf_wtss, color="#2196F3", marker="o", linewidth=2, label="WTSS")

plt.axhline(y=G.number_of_nodes(), color="gray", linestyle="--", alpha=0.5, label=f"|V|={G.number_of_nodes()}")
plt.xlabel("Percentuale del costo totale")
plt.ylabel("|Inf[G, S]|")
plt.title("WTSS — |Inf[G,S]| al variare di k (c_custom)") 
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "wtss_custom", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/8z/67y4_fv93g3dgx30jl7w8ss40000gn/T/ipykernel_1844/3279139263.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Seed set di riferimento calcolati con k=1217
k_ref = 4270
S_ref_f1    = cost_seeds_greedy(G, k_ref, c_custom, f1)
S_ref_f2    = cost_seeds_greedy(G, k_ref, c_custom, f2)
S_ref_f3    = cost_seeds_greedy(G, k_ref, c_custom, f3)

# Range di archi da rimuovere
x_range = list(range(0, 20000, 1000))


inf_f1_edges   = []
inf_f2_edges   = []
inf_f3_edges   = []


for x in x_range:
    G2 = remove_edges(G, x)
    inf_f1_edges.append(len(majority_cascade(G2, S_ref_f1)))
    inf_f2_edges.append(len(majority_cascade(G2, S_ref_f2)))
    inf_f3_edges.append(len(majority_cascade(G2, S_ref_f3)))
    print(f"x={x:5d} | f1={inf_f1_edges[-1]:4d} | f2={inf_f2_edges[-1]:4d} | f3={inf_f3_edges[-1]:4d} |")

x=    0 | f1=3225 | f2=3119 | f3=3532 |
x= 1000 | f1=3210 | f2=3202 | f3=3533 |
x= 2000 | f1=3224 | f2=3140 | f3=3545 |
x= 3000 | f1=3138 | f2=3100 | f3=3535 |
x= 4000 | f1=3123 | f2=3098 | f3=3521 |
x= 5000 | f1=3119 | f2=3129 | f3=3522 |
x= 6000 | f1=3128 | f2=3132 | f3=3519 |
x= 7000 | f1=3127 | f2=3136 | f3=3521 |
x= 8000 | f1=3127 | f2=3129 | f3=3520 |
x= 9000 | f1=3255 | f2=3134 | f3=3522 |
x=10000 | f1=3259 | f2=3138 | f3=3522 |
x=11000 | f1=3259 | f2=3129 | f3=3522 |
x=12000 | f1=3253 | f2=3135 | f3=3521 |
x=13000 | f1=3167 | f2=3121 | f3=3507 |
x=14000 | f1=3159 | f2=3206 | f3=3507 |
x=15000 | f1=3176 | f2=3188 | f3=3502 |
x=16000 | f1=3176 | f2=3186 | f3=3501 |
x=17000 | f1=3175 | f2=3111 | f3=3497 |
x=18000 | f1=3182 | f2=3132 | f3=3501 |
x=19000 | f1=3188 | f2=3129 | f3=3500 |


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(x_range, inf_f1_edges,   color="#E91E63", marker="s", linewidth=2, label="Greedy-f1")
plt.plot(x_range, inf_f2_edges,   color="#FF9800", marker="D", linewidth=2, label="Greedy-f2")
plt.plot(x_range, inf_f3_edges,   color="#9C27B0", marker="^", linewidth=2, label="Greedy-f3")

# Linea di riferimento: |Inf| sul grafo originale senza rimozioni
#f1=3225 | f2=3119 | f3=3532
plt.axhline(y=3225, color="#E91E63", linestyle="--", alpha=0.3)
plt.axhline(y=3119, color="#FF9800", linestyle="--", alpha=0.3)
plt.axhline(y=3532, color="#9C27B0", linestyle="--", alpha=0.3)

plt.xlabel("Archi rimossi (x)")
plt.ylabel("|Inf[G', S]|")
plt.title("Step 2 — Effetto rimozione archi (c_custom, k=4270)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "edge_removal_C.png", dpi=150, bbox_inches="tight")


In [ ]:
k_ref = 4270
S_ref_f1    = cost_seeds_greedy(G, k_ref, c_custom, f1)
S_ref_f2    = cost_seeds_greedy(G, k_ref, c_custom, f2)
S_ref_f3    = cost_seeds_greedy(G, k_ref, c_custom, f3)

# Stesso k di riferimento

y_range = list(range(0, 1000, 50))

inf_f1_nodes   = []
inf_f2_nodes   = []
inf_f3_nodes   = []

for y in y_range:
    G2 = remove_vertices(G, y)
    # I seed rimossi insieme ai nodi vanno esclusi
    S2_f1   = S_ref_f1   & set(G2.nodes())
    S2_f2   = S_ref_f2   & set(G2.nodes())
    S2_f3   = S_ref_f3   & set(G2.nodes())
    
    inf_f1_nodes.append(len(majority_cascade(G2, S2_f1)))
    inf_f2_nodes.append(len(majority_cascade(G2, S2_f2)))
    inf_f3_nodes.append(len(majority_cascade(G2, S2_f3)))
    print(f"y={y:4d} | f1={inf_f1_nodes[-1]:4d} | f2={inf_f2_nodes[-1]:4d} | f3={inf_f3_nodes[-1]:4d}")

y=   0 | f1=3225 | f2=3119 | f3=3532
y=  50 | f1=3181 | f2=3079 | f3=3485
y= 100 | f1=3151 | f2=3048 | f3=3455
y= 150 | f1=3115 | f2=2977 | f3=3410
y= 200 | f1=3070 | f2=2934 | f3=3360
y= 250 | f1=3005 | f2=2946 | f3=3304
y= 300 | f1=3010 | f2=2960 | f3=3263
y= 350 | f1=2966 | f2=2909 | f3=3206
y= 400 | f1=2922 | f2=2912 | f3=3165
y= 450 | f1=2879 | f2=2866 | f3=3120
y= 500 | f1=2844 | f2=2834 | f3=3084
y= 550 | f1=2811 | f2=2753 | f3=3050
y= 600 | f1=2781 | f2=2770 | f3=3007
y= 650 | f1=2765 | f2=2740 | f3=2975
y= 700 | f1=2713 | f2=2688 | f3=2935
y= 750 | f1=2665 | f2=2649 | f3=2891
y= 800 | f1=2607 | f2=2603 | f3=2825
y= 850 | f1=2567 | f2=2563 | f3=2803
y= 900 | f1=2549 | f2=2526 | f3=2767
y= 950 | f1=2512 | f2=2487 | f3=2728


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(x_range, inf_f1_nodes,   color="#E91E63", marker="s", linewidth=2, label="Greedy-f1")
plt.plot(x_range, inf_f2_nodes,   color="#FF9800", marker="D", linewidth=2, label="Greedy-f2")
plt.plot(x_range, inf_f3_nodes,   color="#9C27B0", marker="^", linewidth=2, label="Greedy-f3")

# Linea di riferimento: |Inf| sul grafo originale senza rimozioni
#f1=3225 | f2=3119 | f3=3532
plt.axhline(y=3225, color="#E91E63", linestyle="--", alpha=0.3)
plt.axhline(y=3119, color="#FF9800", linestyle="--", alpha=0.3)
plt.axhline(y=3532, color="#9C27B0", linestyle="--", alpha=0.3)

plt.xlabel("Nodi rimossi (x)")
plt.ylabel("|Inf[G', S]|")
plt.title("Step 2 — Effetto rimozione nodi (c_custom, k=4270)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "node_removal_C.png", dpi=150, bbox_inches="tight")
